In [1]:
import re
from pathlib import Path

import pandas as pd

In [2]:
# Configuration: folder with the .txt files and folder for the generated .csv files
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("./results/")
PATTERN = "results_*.txt"

In [3]:
# Regex matching one result line, e.g.:
# Description: [Course = '9500', Gender = '0'], Target: Target = 'Graduate' ; Rank : 5
LINE_RE = re.compile(
    r"^Description:\s*\[(?P<Description>.*)\],\s*Target:\s*(?P<Target>.*?)\s*;"
    r"\s*Rank\s*:\s*(?P<Rank>\d+)\s*$"
)

In [4]:
def parse_file(path):
    """Parse a txt file and return a DataFrame sorted by Rank (ascending).

    The index keeps the original line order of the file (0-based).
    """
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip("\r\n")  # keep inner tabs, they are part of some attribute names
            if not line.strip():
                continue  # skip empty lines
            match = LINE_RE.match(line.strip())
            if match is None:
                print(f"Warning: skipping unparsed line in {path.name}: {line[:80]}")
                continue
            d = match.groupdict()
            rows.append({
                "Description": d["Description"],
                "Rank": int(d["Rank"]),
            })

    df = pd.DataFrame(rows, columns=["Description", "Rank"])
    # Stable sort so ties keep their original order
    return df.sort_values("Rank", ascending=True, kind="stable")

In [5]:
# Convert every matching txt file into a csv named [algorithm]_[dataset].csv
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for txt_path in sorted(INPUT_DIR.glob(PATTERN)):
    # Expected name: results_[algorithm]_[id]_[dataset].txt
    # maxsplit=3 keeps any underscores in the dataset name intact
    parts = txt_path.stem.split("_", 3)
    if len(parts) != 4 or parts[0] != "results":
        print(f"Warning: skipping {txt_path.name}, unexpected file name")
        continue
    _, algorithm, _, dataset = parts

    df = parse_file(txt_path)
    csv_path = OUTPUT_DIR / f"{algorithm}_{dataset}.csv"
    df.to_csv(csv_path)  # the index is written as an unnamed first column
    print(f"{txt_path.name} -> {csv_path.name} ({len(df)} rows)")

results_BerryFinder_35018_Depression.txt -> BerryFinder_Depression.csv (2 rows)
results_BerryFinder_3909_Mushrooms.txt -> BerryFinder_Mushrooms.csv (109 rows)
results_BerryFinder_49651_Vancomycin.txt -> BerryFinder_Vancomycin.csv (11 rows)
results_BerryFinder_57971_Cardio.txt -> BerryFinder_Cardio.csv (2 rows)
results_BerryFinder_62772_StudentSuccess.txt -> BerryFinder_StudentSuccess.csv (84 rows)
results_BerryFinder_64164_Car.txt -> BerryFinder_Car.csv (2 rows)
results_BerryFinder_72300_connect4.txt -> BerryFinder_connect4.csv (35 rows)
results_BerryFinder_74196_MIMIC-IV.txt -> BerryFinder_MIMIC-IV.csv (3 rows)
results_BerryFinder_75864_Chess.txt -> BerryFinder_Chess.csv (9 rows)
results_BerryFinder_95805_Dermatology.txt -> BerryFinder_Dermatology.csv (177 rows)
